# Colab 33 — SNNEED (regression + pooling) vs ESM-2 vs Dice

Headline comparison for the **pivoted** architecture: the deployed SNNEED is now **regression + pooling**
(`reg·pool` — no classifier head; the encoder is trained with band-weighted MSE and the readout `1−‖e_a−e_b‖/2`
*is* the retrieval score). We re-establish the method comparison with this encoder against the two baselines
that matter for the algorithm-approximation story:

- **SNNEED (reg·pool)** — small task-specific edit-distance encoder (ours).
- **ESM-2** — frozen 35M protein LM, task-agnostic (Fenoy: PLM cosine carries a similarity signal).
- **Dice** — classical, no learning, length-normalized 3-gram overlap.

*(trigram-count and length-only dropped, per the pivot.)* Evaluated on **synth / 3Di / SS / AA** with
**Spearman / AUROC / MAP@10**. Encoder-based methods score by cosine; Dice by shared-3-gram Dice. Ground truth
= exact normLev on our own strings. Reads no result CSVs; writes receipts only.

**AUROC is on the stratified pair set** (≥0.70 vs the decile-balanced background) — internally consistent
across all three methods here; not the full-pool colab29b AUROC. **MAP@10 is full-pool** de-hubbed retrieval.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib transformers --quiet

In [ ]:
import time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

N_TRAIN = 30_000; SEEDS = [0, 1, 2]; EPOCHS = 30
STRAT_PER_BIN = 400; STRAT_CAND = 200_000
SYN_PERTURB, SYN_INDEP = 20_000, 8_000
ESM2_MODEL = 'facebook/esm2_t12_35M_UR50D'
FEED_ORDER   = ['synth', '3Di', 'SS', 'AA']
METHOD_ORDER = ['SNNEED', 'ESM-2', 'Dice']
METHOD_COLOR = {'SNNEED': '#c026a6', 'ESM-2': '#2ca02c', 'Dice': '#9e9e9e'}

## 2. Constants, helpers, and the reg·pool encoder

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc, rng): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))

class EncPool(nn.Module):
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class RegModel(nn.Module):   # DEPLOYED SNNEED: regression + pooling, no head
    def __init__(s, enc): super().__init__(); s.encoder = enc
    def forward(s, a, b):
        ea, eb = s.encoder(a), s.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0
class DS(Dataset):
    def __init__(s, pp): s.p = pp
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), torch.tensor(l, dtype=torch.float32)
def band_w(y):
    w = torch.full_like(y, 2.0); w[y < BAND_LOW_AA] = 0.5; w[y >= BAND_HIGH] = 4.0; return w
def build_pairs(n, seed):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs
def train_regpool(pairs, seed):
    torch.manual_seed(seed); model = RegModel(EncPool()).to(device)
    dl = DataLoader(DS(pairs), batch_size=BS, shuffle=True); opt = torch.optim.Adam(model.parameters(), 1e-3)
    model.train()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            pred = model(a, b); loss = (band_w(y) * (pred - y)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'    [reg·pool s={seed}] epoch {ep}/{EPOCHS} MSE {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); return model

## 3. Pools + oracles + stratified pairs + synth feed (built once)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SEQ = {f: list(LOOK[f].values()) for f in LOOK}
CATH_FEEDS = ['AA', 'SS', '3Di']
for f in CATH_FEEDS: print(f'  {f:<4} pool = {len(POOL_SEQ[f]):>6}')

def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = np.array([len(s) for s in seqs]); N = len(seqs); T_high = {}; pos = []
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0; hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            for j in hi:
                if j > i: pos.append((i, int(j), float(row[j])))
    return dict(T_high=T_high, pos_pairs=pos)
ORACLE = {}
for f in CATH_FEEDS:
    print(f'oracle {f} (SS slow)...'); ORACLE[f] = build_oracle(f)
    print(f'  {f}: queries@0.70={len(ORACLE[f]["T_high"])}, pos pairs={len(ORACLE[f]["pos_pairs"])}')

def build_strat_pairs(feed, rng):
    seqs = POOL_SEQ[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND); keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if ORACLE[feed]['pos_pairs']:
        pa = np.array(ORACLE[feed]['pos_pairs'], float)
        a = np.concatenate([a, pa[:, 0].astype(np.int64)]); b = np.concatenate([b, pa[:, 1].astype(np.int64)]); nl = np.concatenate([nl, pa[:, 2]])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: t = rng.permutation(idx)[:STRAT_PER_BIN]; ai.append(a[t]); aj.append(b[t]); av.append(nl[t])
    return dict(i=np.concatenate(ai).astype(np.int64), j=np.concatenate(aj).astype(np.int64), nl=np.concatenate(av))
STRAT = {f: build_strat_pairs(f, np.random.default_rng(999)) for f in CATH_FEEDS}

def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=20260810):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r); part = perturb(base, int(r.integers(0, len(base)+1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(n_indep): recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]; nl = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, l = recs[int(idx)]; I.append(len(seqs)); seqs.append(a); J.append(len(seqs)); seqs.append(b); NL.append(l)
    return seqs, np.array(I), np.array(J), np.array(NL)
SYN_SEQ, SYN_I, SYN_J, SYN_NL = build_synth_feed(SYN_PERTURB, SYN_INDEP)
POOL_SEQ['synth'] = SYN_SEQ; ORACLE['synth'] = build_oracle('synth')
print(f'synth: pool={len(SYN_SEQ)}, queries@0.70={len(ORACLE["synth"]["T_high"])}')

## 4. Scorers — embeddings (SNNEED/ESM-2) and Dice

In [ ]:
def _auroc(sim, nl):
    y = (nl >= BAND_HIGH).astype(int)
    return roc_auc_score(y, sim) if 0 < y.sum() < len(y) else np.nan
def map10_emb(E_t, T_high, k=10, qb=256):
    q = list(T_high.keys())
    if not q: return np.nan
    aps = []
    for s0 in range(0, len(q), qb):
        qi = q[s0:s0+qb]; sc = E_t[qi] @ E_t.t()
        for r, idx in enumerate(qi): sc[r, idx] = -1e9
        top = torch.topk(sc, k, dim=1).indices.cpu().numpy()
        for r, idx in enumerate(qi):
            ts = set(T_high[idx].tolist()); hits = 0; ap = 0.0
            for rr, o in enumerate(top[r], 1):
                if o in ts: hits += 1; ap += hits / rr
            aps.append(ap / min(len(ts), k))
    return float(np.mean(aps))
def _pairs(feed):
    if feed == 'synth': return SYN_I, SYN_J, SYN_NL
    P = STRAT[feed]; return P['i'], P['j'], P['nl']
def eval_emb(E_np, feed):
    I, J, nl = _pairs(feed); sim = np.sum(E_np[I] * E_np[J], axis=1)
    Et = torch.as_tensor(E_np, device=device)
    return spearmanr(sim, nl).correlation, _auroc(sim, nl), map10_emb(Et, ORACLE[feed]['T_high'])

# --- Dice over binary 3-grams ---
def build_kmer(feed, k=3):
    seqs = POOL_SEQ[feed]; vocab = {}; rows = []; cols = []
    for i, s in enumerate(seqs):
        for g in set(s[t:t+k] for t in range(len(s)-k+1)):
            rows.append(i); cols.append(vocab.setdefault(g, len(vocab)))
    B = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(seqs), max(1, len(vocab))))
    return B, np.asarray(B.sum(1)).ravel()
def dice_pair(B, sz, I, J):
    inter = np.asarray(B[I].multiply(B[J]).sum(1)).ravel().astype(float)
    return 2 * inter / np.maximum(sz[I] + sz[J], 1e-9)
def eval_dice(feed):
    B, sz = build_kmer(feed); I, J, nl = _pairs(feed); sim = dice_pair(B, sz, I, J)
    T = ORACLE[feed]['T_high']; aps = []
    for qi in T:
        inter = np.asarray(B.dot(B[qi].T).todense()).ravel().astype(float)
        s = 2 * inter / np.maximum(sz + sz[qi], 1e-9); s[qi] = -1.0
        order = np.argpartition(-s, min(10, len(s)-1))[:10]; order = order[np.argsort(-s[order])]
        ts = set(T[qi].tolist()); hits = 0; ap = 0.0
        for rr, o in enumerate(order, 1):
            if o in ts: hits += 1; ap += hits / rr
        aps.append(ap / min(len(ts), 10))
    return spearmanr(sim, nl).correlation, _auroc(sim, nl), (float(np.mean(aps)) if aps else np.nan)

@torch.no_grad()
def snn_embed(model, feed, bs=256):
    seqs = POOL_SEQ[feed]; out = []
    for i in range(0, len(seqs), bs):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+bs]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

_esm = {}
@torch.no_grad()
def esm_embed(feed, bs=32):
    cf = f'colab33_esm2_{feed}.npy'
    if os.path.exists(cf): return np.load(cf)
    if 'mdl' not in _esm:
        from transformers import AutoTokenizer, AutoModel
        _esm['tok'] = AutoTokenizer.from_pretrained(ESM2_MODEL)
        _esm['mdl'] = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
    tok, mdl = _esm['tok'], _esm['mdl']; seqs = POOL_SEQ[feed]
    order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]; batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = mdl(**enc).last_hidden_state; mask = enc['attention_mask'].clone(); mask[:, 0] = 0
        for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0
        m = mask.unsqueeze(-1).float(); e = F.normalize((h*m).sum(1)/m.sum(1).clamp(min=1), dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    E = np.stack(out).astype(np.float32); np.save(cf, E); return E

## 5a. SNNEED (reg·pool) — train per seed, evaluate

In [ ]:
rows = []
for seed in SEEDS:
    pairs = build_pairs(N_TRAIN, seed); model = train_regpool(pairs, seed)
    for feed in FEED_ORDER:
        sp, au, mp = eval_emb(snn_embed(model, feed), feed)
        rows.append(dict(method='SNNEED', seed=seed, feed=feed, spearman=sp, auroc=au, map10=mp))
    print(f'  SNNEED s={seed}: ' + '  '.join(f'{d["feed"]}:MAP={d["map10"]:.2f}' for d in rows[-len(FEED_ORDER):]))

## 5b. ESM-2 (frozen; embeds each pool once, cached)

In [ ]:
for feed in FEED_ORDER:
    sp, au, mp = eval_emb(esm_embed(feed), feed)
    rows.append(dict(method='ESM-2', seed=0, feed=feed, spearman=sp, auroc=au, map10=mp))
    print(f'  ESM-2 {feed}: rho={sp:.2f} AUROC={au:.2f} MAP={mp:.2f}')

## 5c. Dice (classical, no learning)

In [ ]:
for feed in FEED_ORDER:
    sp, au, mp = eval_dice(feed)
    rows.append(dict(method='Dice', seed=0, feed=feed, spearman=sp, auroc=au, map10=mp))
    print(f'  Dice {feed}: rho={sp:.2f} AUROC={au:.2f} MAP={mp:.2f}')
res = pd.DataFrame(rows); res.to_csv('colab33_metrics.csv', index=False)

## 6. Tables — method × feed, per metric (SNNEED mean over seeds)

In [ ]:
METRICS = [('spearman', 'Spearman'), ('auroc', 'AUROC(>=0.70, stratified)'), ('map10', 'MAP@10 (full-pool)')]
present = [m for m in METHOD_ORDER if m in set(res.method)]
for col, name in METRICS:
    tab = res.groupby(['method', 'feed'])[col].mean().unstack('feed').reindex(index=present, columns=FEED_ORDER)
    print('=' * 60); print(name); print('=' * 60); print(tab.round(3).to_string()); print()

## 7. Figure — SNNEED vs ESM-2 vs Dice across feeds

In [ ]:
import matplotlib.pyplot as plt
def despine(ax): ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
present = [m for m in METHOD_ORDER if m in set(res.method)]
xpos = np.arange(len(FEED_ORDER)); w = 0.8 / max(1, len(present))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (col, name) in zip(axes, METRICS):
    for c, meth in enumerate(present):
        m = [res[(res.method == meth) & (res.feed == f)][col].mean() for f in FEED_ORDER]
        sd = [res[(res.method == meth) & (res.feed == f)][col].std() for f in FEED_ORDER]
        ax.bar(xpos + (c - (len(present)-1)/2) * w, m, w, yerr=sd, capsize=2,
               color=METHOD_COLOR[meth], label=meth)
    ax.set_xticks(xpos); ax.set_xticklabels(FEED_ORDER); ax.set_title(name)
    ax.set_ylim(bottom=min(0, ax.get_ylim()[0])); despine(ax)
axes[0].legend(fontsize=9)
plt.tight_layout(); plt.savefig('colab33_regpool_vs_baselines.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. How to read this

The deployed SNNEED is **regression + pooling** — no classifier head. The story to check against the deck:
**SNNEED ≥ ESM-2 on the task-specific job (Spearman/MAP), and it transfers to the structural alphabets
(3Di/SS)** while ESM-2 saturates on high-similarity pairs (Fenoy's fingerprint). Dice is the classical floor.

- **synth** = in-distribution reference; **AA** = the noise-level control (few natural high-sim pairs — read
  cautiously); **SS/3Di** = the cross-representation transfer (encoder trained only on synthetic AA).
- SNNEED shows mean ± std over the seeds; ESM-2/Dice are deterministic.
- Compare these SNNEED numbers to the colab29b classifier SNN to confirm the pivot loses nothing on retrieval
  (and colab32's RMSE shows it *gains* value fidelity).

Outputs (written, not read): `colab33_metrics.csv`, `colab33_regpool_vs_baselines.png`, `colab33_esm2_*.npy`.